In [1]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load the original dataset
DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)
df.head()


Original dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:

# Create a separate working copy
df_clean = df.copy()

# Convert blank or whitespace-only values to NaN
df_clean["TotalCharges"] = df_clean["TotalCharges"].replace(
    r"^\s*$", np.nan, regex=True
)

# Convert the column from text to numeric
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

print("TotalCharges dtype:", df_clean["TotalCharges"].dtype)
print("Missing TotalCharges:", df_clean["TotalCharges"].isna().sum())

# Inspect the affected customers
df_clean.loc[
    df_clean["TotalCharges"].isna(),
    ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]


TotalCharges dtype: float64
Missing TotalCharges: 11


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


In [3]:

# Confirm all missing TotalCharges values belong to zero-tenure customers
missing_mask = df_clean["TotalCharges"].isna()

print(
    "All missing rows have zero tenure:",
    df_clean.loc[missing_mask, "tenure"].eq(0).all()
)

# Replace their missing total charges with zero
df_clean.loc[missing_mask, "TotalCharges"] = 0

print("Remaining missing TotalCharges:", df_clean["TotalCharges"].isna().sum())
print("TotalCharges dtype:", df_clean["TotalCharges"].dtype)


All missing rows have zero tenure: True
Remaining missing TotalCharges: 0
TotalCharges dtype: float64


In [4]:

# Preserve customer IDs for identifying records later
customer_ids = df_clean["customerID"].copy()

# Preserve churn separately for post-clustering interpretation
churn_labels = df_clean["Churn"].copy()

# Remove identifiers and churn from clustering features
X = df_clean.drop(
    columns=["customerID", "Churn"]
)

print("Feature matrix shape:", X.shape)
print("\nRemaining columns:")
print(X.columns.tolist())

print("\nChurn labels shape:", churn_labels.shape)
print("Customer IDs shape:", customer_ids.shape)


Feature matrix shape: (7043, 19)

Remaining columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Churn labels shape: (7043,)
Customer IDs shape: (7043,)


In [5]:

# Identify numerical and categorical columns
numerical_columns = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical features:", numerical_columns)
print("\nNumber of numerical features:", len(numerical_columns))

print("\nCategorical features:", categorical_columns)
print("\nNumber of categorical features:", len(categorical_columns))


Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Number of numerical features: 4

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Number of categorical features: 15


In [6]:

# Build the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_columns
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


In [7]:

# Fit the transformer and transform the feature matrix
X_processed = preprocessor.fit_transform(X)

# Convert the processed array into a DataFrame
feature_names = preprocessor.get_feature_names_out()

X_processed = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=X.index
)

print("Processed matrix shape:", X_processed.shape)
print("\nMissing values:", X_processed.isna().sum().sum())
print("\nAll features numeric:", X_processed.dtypes.apply(
    lambda dtype: np.issubdtype(dtype, np.number)
).all())

X_processed.head()


Processed matrix shape: (7043, 45)

Missing values: 0

All features numeric: True


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Female,cat__gender_Male,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,cat__Dependents_Yes,...,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
0,-0.439916,-1.277445,-1.160323,-0.992611,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,-0.439916,0.066327,-0.259629,-0.172165,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,-0.439916,-1.236724,-0.362660,-0.958066,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,-0.439916,0.514251,-0.746535,-0.193672,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,-0.439916,-1.236724,0.197365,-0.938874,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [8]:

# Inspect the processed feature names
print("Processed feature names:")
print(X_processed.columns.tolist())

# Check numerical feature means and standard deviations
print("\nScaled numerical feature means:")
print(X_processed[
    [f"num__{column}" for column in numerical_columns]
].mean().round(3))

print("\nScaled numerical feature standard deviations:")
print(X_processed[
    [f"num__{column}" for column in numerical_columns]
].std().round(3))

# Check for infinite values
print(
    "\nContains infinite values:",
    np.isinf(X_processed.to_numpy()).any()
)


Processed feature names:
['num__SeniorCitizen', 'num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'cat__gender_Female', 'cat__gender_Male', 'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No', 'cat__Dependents_Yes', 'cat__PhoneService_No', 'cat__PhoneService_Yes', 'cat__MultipleLines_No', 'cat__MultipleLines_No phone service', 'cat__MultipleLines_Yes', 'cat__InternetService_DSL', 'cat__InternetService_Fiber optic', 'cat__InternetService_No', 'cat__OnlineSecurity_No', 'cat__OnlineSecurity_No internet service', 'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No', 'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes', 'cat__DeviceProtection_No', 'cat__DeviceProtection_No internet service', 'cat__DeviceProtection_Yes', 'cat__TechSupport_No', 'cat__TechSupport_No internet service', 'cat__TechSupport_Yes', 'cat__StreamingTV_No', 'cat__StreamingTV_No internet service', 'cat__StreamingTV_Yes', 'cat__StreamingMovies_No', 'cat__StreamingMovies_No internet service', 'cat

In [9]:

from pathlib import Path

# Define the processed-data directory
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save processed clustering features
X_processed.to_csv(
    OUTPUT_DIR / "customer_features.csv",
    index=False
)

# Save customer IDs and churn labels separately for interpretation
metadata = pd.DataFrame({
    "customerID": customer_ids,
    "Churn": churn_labels
})

metadata.to_csv(
    OUTPUT_DIR / "customer_metadata.csv",
    index=False
)

print("Saved files:")
print(OUTPUT_DIR / "customer_features.csv")
print(OUTPUT_DIR / "customer_metadata.csv")


Saved files:
../data/processed/customer_features.csv
../data/processed/customer_metadata.csv


In [10]:

# Reload the saved files to verify they exist and are readable
saved_features = pd.read_csv(
    OUTPUT_DIR / "customer_features.csv"
)

saved_metadata = pd.read_csv(
    OUTPUT_DIR / "customer_metadata.csv"
)

print("Saved features shape:", saved_features.shape)
print("Saved metadata shape:", saved_metadata.shape)

print(
    "Row counts match:",
    len(saved_features) == len(saved_metadata)
)

print(
    "Churn column included in features:",
    "Churn" in saved_features.columns
)


Saved features shape: (7043, 45)
Saved metadata shape: (7043, 2)
Row counts match: True
Churn column included in features: False


## Summary and Key Findings

### Data Cleaning
- Loaded 7,043 customer records with 21 original columns.
- Converted `TotalCharges` from string to numeric.
- Handled 11 blank `TotalCharges` values associated with zero-tenure customers by assigning zero total charges.
- Removed `customerID` and `Churn` from the clustering feature matrix.

### Feature Engineering and Transformation
- Identified 4 numerical and 15 categorical input features.
- Applied `StandardScaler` to numerical features.
- Applied `OneHotEncoder` to categorical features.
- Transformed the original 19 input features into 45 numerical features.

### Validation
- All 7,043 records were retained.
- No missing values remained in the processed feature matrix.
- All processed features were numerical.
- Saved the feature matrix and customer metadata as separate CSV files.

### Output Files
- `data/processed/customer_features.csv`
- `data/processed/customer_metadata.csv`

### Important Decision
The `Churn` label was excluded from clustering inputs to prevent target leakage. It will be used later to interpret the observed churn rates of the resulting customer segments.

### Next Steps
Apply K-Means clustering, investigate an appropriate number of clusters, and evaluate cluster quality using the elbow method and silhouette score.